In [1]:
import matlab.engine
import pickle
import os
import torch
import json
from mylib import yolo_patch
from mylib import utils
import math
import numpy as np
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
import skimage.io as io
from ultralytics import YOLO 
from dotenv import load_dotenv
from tqdm import tqdm  # For progress tracking
import time
load_dotenv()  # This loads from .env in the current directory

True

In [2]:
# Load annotations for both datasets
cocos = {
    "train2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_train2017.json")),
    "val2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_val2017.json")),
}

loading annotations into memory...
Done (t=13.69s)
creating index...
index created!
loading annotations into memory...
Done (t=0.47s)
creating index...
index created!


In [3]:
# YOLO ID -- Class mapping from json file
with open('indoor_objects.json') as f:
    indoor_objects = json.load(f)
    indoor_objects = {k: v for d in indoor_objects['indoor_classes'] for k, v in d.items()}

# Bin range -- Class from json file
with open("out_bins_per_class.json") as f:
    out_bins_per_class = json.load(f)

# YOLO IDs -- COCO IDs mapping
train_coco = list(cocos.values())[0]
all_cats = train_coco.loadCats(train_coco.getCatIds())
yolo_id_to_coco_id = {i: cat["id"] for i, cat in enumerate(sorted(all_cats, key=lambda x: x["id"]))} # eg. Yolo ID 79 -> COCO ID 90

# Build lists 
yolo_ids = [int(k) for k in indoor_objects.keys()]
coco_ids = [yolo_id_to_coco_id[int(k)] for k in indoor_objects.keys()]
classes_names = [v for v in indoor_objects.values()]
classes_bins = [out_bins_per_class[v] for v in indoor_objects.values()]
num_classes = len(classes_bins)

# Print information
print("Number of classes:", num_classes,'\n')
print("YOLO IDs:", yolo_ids)
print("COCO IDs:", coco_ids)
print("Classes names:", classes_names)
print("Classes bins:", classes_bins)


Number of classes: 27 

YOLO IDs: [26, 39, 40, 41, 42, 43, 44, 45, 47, 56, 57, 58, 59, 60, 61, 62, 63, 65, 67, 68, 69, 70, 71, 73, 74, 75, 77]
COCO IDs: [31, 44, 46, 47, 48, 49, 50, 51, 53, 62, 63, 64, 65, 67, 70, 72, 73, 75, 77, 78, 79, 80, 81, 84, 85, 86, 88]
Classes names: ['handbag', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'apple', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'remote', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'book', 'clock', 'vase', 'teddy bear']
Classes bins: [[0.0, 0.05, 0.11, 0.19, 0.33, 0.53, 0.84, 1.41, 2.44, 4.87, 99.4], [0.0, 0.06, 0.11, 0.17, 0.25, 0.37, 0.55, 0.87, 1.52, 3.39, 99.01], [0.0, 0.09, 0.17, 0.26, 0.41, 0.65, 1.03, 1.73, 3.2, 7.09, 98.2], [0.0, 0.07, 0.13, 0.22, 0.34, 0.54, 0.89, 1.55, 2.78, 5.78, 100.0], [0.0, 0.11, 0.23, 0.4, 0.62, 0.99, 1.64, 2.71, 4.88, 9.2, 99.45], [0.0, 0.04, 0.09, 0.17, 0.3, 0.55, 0.94, 1.59, 3.0, 6.7, 83.22], [0.0, 0.06, 0.12, 0.21, 0.34, 0.54, 

In [4]:
# === Load YOLO model ===
yolo_model = YOLO("yolov8n.pt")  # Swap with yolov8s.pt or yolov8x.pt as needed

In [5]:
# === Helper: Get COCO split for a given image ID ===
def find_split_for_image(img_id):
    for split, coco_obj in cocos.items():
        if img_id in coco_obj.imgs:
            return split, coco_obj
    return None, None

In [8]:
# === IoU threshold for filtering ===
IOU_THRESHOLD = 0.5

# === Data structure to store the vectors scores ===
vector_scores_struct = {cls: {b: [] for b in range(len(classes_bins[0]))} for cls in classes_names}

# Loop through every class_id
for i in range(num_classes):

    # Get all image IDs for this class (from both splits)
    img_ids = []
    for split, coco_obj in cocos.items():
        img_ids += coco_obj.getImgIds(catIds=[coco_ids[i]])

    k = 0
    # Loop through each image ID
    for img_id in tqdm(img_ids, desc=f"Processing {len(img_ids)} images for {classes_names[i]}", unit="image"):
        
        # Find the split for the current image ID
        split, any_coco = find_split_for_image(img_id)
        
        # Load the image 
        img_data = any_coco.imgs[img_id]
        img_path = os.path.join(os.getenv("COCO_DATA"), split, img_data["file_name"])
        img = io.imread(img_path)

        # Plot ground truth boxes from COCO using COCO API
        ann_ids = any_coco.getAnnIds(imgIds=img_id, catIds=[coco_ids[i]], iscrowd=None)
        anns = any_coco.loadAnns(ann_ids)

        if img.ndim == 2:  # grayscale
            img = np.stack([img]*3, axis=-1)

        # Prediction
        results = yolo_model.predict(source=img, device='cuda:0', conf=0.25, iou=0.45, verbose=False, max_det=100)
        
        # Unpack results
        result = results[0] # One image, one result
        xyxy = result.boxes.xyxy
        xywh = result.boxes.xywh
        conf = result.boxes.conf
        cls = result.boxes.cls
        prob_vectors = result.boxes.data[:, 6:]  # Probability vectors for each class

        # Plot the image
        # plt.figure(figsize=(5, 5))
        # plt.imshow(img)
        # plt.axis('off')
        # plt.title(f"Image ID: {img_id}, Ground-truth Class: {classes_names[i]}")

        # Loop ground truth annotations
        for ann in anns:
            bbox = ann["bbox"]
            x1_gt, y1_gt, w_gt, h_gt = bbox
            x2_gt = x1_gt + w_gt
            y2_gt = y1_gt + h_gt

            # Compute the scale of ground truth bounding box compared to whole image
            scale_gt = w_gt * h_gt / (img.shape[0] * img.shape[1])*100
            # print(f"Ground truth Scale: {scale_gt:.4f}%")

            # Compute bin index
            bin_idx = utils.bin_index(scale_gt, classes_bins[i])
            # print(f"Ground truth Bin index: {bin_idx}")
            # print("Classes bins:", classes_bins[i])

            # Plot ground truth boxes
            # plt.gca().add_patch(plt.Rectangle((x1_gt, y1_gt), w_gt, h_gt, fill=False, edgecolor="green", linewidth=2))

            # Loop through every detection
            for box_xyxy, box_xywh, c, cl, prob_v in zip(xyxy, xywh, conf, cls, prob_vectors):
                x1, y1, x2, y2 = box_xyxy.tolist()
                cx, cy, w, h = box_xywh.tolist()
                confidence = c.item()
                class_id = int(cl.item()) 
                prob_vector = prob_v.tolist()

                # Compute IoU
                IoU = utils.compute_iou([x1, y1, x2, y2], [x1_gt, y1_gt, x2_gt, y2_gt])

                # Check if IoU is above the threshold
                if IoU > IOU_THRESHOLD:

                    # Add vector probability to the corresponding bin
                    vector_scores_struct[classes_names[i]][bin_idx].append(prob_vector)

                    # Print results
                    # print(f"XYXY (corner format): ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
                    # print(f"XYWH (center format): ({cx:.1f}, {cy:.1f}, {w:.1f}, {h:.1f})")
                    # print(f"Confidence: {confidence:.4f}, Class ID: {class_id}, Class Name: {yolo_model.names[class_id]}")
                    # print(f"Probability vector: {[f'{x:.3e}' for x in prob_vector]}")
                    # print(f"IoU: {IoU:.4f}")
                    # print("---")

                    # # Plot predicted boxes
                    # plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=1))
                    # plt.text(x1, y1, f"{yolo_model.names[class_id]} {confidence:.2f}", color="red", fontsize=12)
                
        # plt.show()

        # # Increment
        # k +=1

        # if k > 5:
        #     break


Processing 2234 images for teddy bear: 100%|██████████| 2234/2234 [03:20<00:00, 11.16image/s]


In [9]:
# Save to pickle
with open("out_vector_scores_struct.pkl", "wb") as f:
    pickle.dump(vector_scores_struct, f)

In [10]:
# Convert the vector scores structure to numpy arrays
for cls in vector_scores_struct:
    for b in vector_scores_struct[cls]:
        vector_scores_struct[cls][b] = np.vstack(vector_scores_struct[cls][b]) if vector_scores_struct[cls][b] else np.empty((0, 80))

In [11]:
# Print the number of vectors in each bin
for cls in vector_scores_struct:
    for b in vector_scores_struct[cls]:
        count = len(vector_scores_struct[cls][b]) if isinstance(vector_scores_struct[cls][b], list) else vector_scores_struct[cls][b].shape[0]
        print(f"{cls} - bin {b}: {count} vectors")


handbag - bin 0: 2 vectors
handbag - bin 1: 0 vectors
handbag - bin 2: 13 vectors
handbag - bin 3: 39 vectors
handbag - bin 4: 93 vectors
handbag - bin 5: 140 vectors
handbag - bin 6: 211 vectors
handbag - bin 7: 258 vectors
handbag - bin 8: 329 vectors
handbag - bin 9: 358 vectors
handbag - bin 10: 451 vectors
bottle - bin 0: 1 vectors
bottle - bin 1: 12 vectors
bottle - bin 2: 99 vectors
bottle - bin 3: 286 vectors
bottle - bin 4: 550 vectors
bottle - bin 5: 728 vectors
bottle - bin 6: 819 vectors
bottle - bin 7: 1074 vectors
bottle - bin 8: 1218 vectors
bottle - bin 9: 1428 vectors
bottle - bin 10: 1632 vectors
wine glass - bin 0: 0 vectors
wine glass - bin 1: 3 vectors
wine glass - bin 2: 24 vectors
wine glass - bin 3: 99 vectors
wine glass - bin 4: 201 vectors
wine glass - bin 5: 302 vectors
wine glass - bin 6: 401 vectors
wine glass - bin 7: 523 vectors
wine glass - bin 8: 534 vectors
wine glass - bin 9: 571 vectors
wine glass - bin 10: 622 vectors
cup - bin 0: 0 vectors
cup - bi

In [12]:
# Print the whole structure
print(vector_scores_struct)

{'handbag': {0: array([[ 0.00015075,  0.00039845,  1.3031e-06,  6.5733e-06,  1.2734e-06,   1.736e-08,  4.5717e-08,   9.465e-09,  4.6968e-07,  1.5425e-06,  4.6772e-07,  3.4727e-06,  3.4996e-07,  1.4662e-05,  0.00053385,     0.44322,   0.0022824,  0.00089343,  0.00018846,  3.7416e-05,  1.0496e-05,  8.7255e-05,  0.00012959,
         5.3903e-05,  0.00031329,  0.00013413,  0.00089789,  2.9547e-05,  0.00021158,  6.1086e-06,  5.2822e-06,   1.582e-06,   1.011e-05,  7.0866e-07,  3.9425e-07,  5.8435e-06,  8.6526e-06,  4.2995e-06,  0.00011846,  5.3975e-06,  7.5442e-06,  2.6041e-05,  3.6467e-06,  4.1627e-06,  5.7341e-06,  4.6262e-05,
         5.5121e-05,  9.7202e-06,  6.2661e-06,  1.3326e-05,  2.8556e-05,  2.3682e-07,  1.4033e-06,  4.0083e-05,  2.7758e-05,  2.7699e-05,  0.00014778,  5.1327e-05,  9.3577e-05,  6.2165e-05,  5.0701e-06,  3.6729e-05,  5.1881e-06,  0.00035605,  6.7501e-05,  1.3367e-05,  0.00019558,  2.5419e-06,  7.1873e-06,
         3.5886e-06,  3.7928e-05,  0.00060035,  2.8393e-06,  5.

In [ ]:
# eng = matlab.engine.start_matlab()
# eng.addpath(os.getenv("FASTFIT_TOOLBOX"), nargout=0)

# eng.quit()